In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "amazon/chronos-2"

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE
)

print("Chronos-2 loaded successfully!")

In [ ]:
DATA_PATH = "data/parking.csv"
TIMESTAMP_COL = "timestamp"
TARGET_COL = "occupancy_rate"

df = pd.read_csv(DATA_PATH)
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

print(f"Number of rows: {len(df)}")
display(df.head())
display(df[[TIMESTAMP_COL, TARGET_COL]].describe(include="all"))

In [ ]:
series_df = df[[TIMESTAMP_COL, TARGET_COL]].copy()
series_df[TARGET_COL] = pd.to_numeric(series_df[TARGET_COL], errors="coerce")
series_df = series_df.dropna().sort_values(TIMESTAMP_COL).reset_index(drop=True)

print("Missing values:", series_df.isna().sum().to_dict())
print("Min occupancy:", series_df[TARGET_COL].min())
print("Max occupancy:", series_df[TARGET_COL].max())

if not series_df[TARGET_COL].between(0, 1).all():
    print("Warning: occupancy is not fully within 0~1. Please confirm whether the unit is percentage.")

display(series_df.head())

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(series_df[TIMESTAMP_COL], series_df[TARGET_COL], linewidth=1)
plt.title("Historical Parking Occupancy Rate")
plt.xlabel("Time")
plt.ylabel("Occupancy rate")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
PREDICTION_LENGTH = 12
PARKING_ID = "parking_001"

context_df = pd.DataFrame({
    "id": PARKING_ID,
    "timestamp": series_df[TIMESTAMP_COL],
    "target": series_df[TARGET_COL]
})

prediction_df = pipeline.predict_df(
    context_df,
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="id",
    timestamp_column="timestamp",
    target="target"
)

display(prediction_df.tail(PREDICTION_LENGTH))

In [ ]:
print("Forecast result columns:", prediction_df.columns.tolist())

future_df = prediction_df.tail(PREDICTION_LENGTH).copy()
median_candidates = ["0.5", "0.5_quantile", "target", "predictions"]
MEDIAN_COL = next((c for c in median_candidates if c in future_df.columns), None)

if MEDIAN_COL is None:
    numeric_cols = future_df.select_dtypes(include="number").columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric forecast column found. Check prediction_df.columns.")
    MEDIAN_COL = numeric_cols[-1]
    print(f"Median column not identified, falling back to numeric column: {MEDIAN_COL}")

plt.figure(figsize=(14, 4))
plt.plot(series_df[TIMESTAMP_COL].tail(100), series_df[TARGET_COL].tail(100), label="History")
plt.plot(future_df["timestamp"], future_df[MEDIAN_COL], marker="o", label="Chronos-2 forecast")
plt.title("Parking Occupancy Forecast")
plt.xlabel("Time")
plt.ylabel("Occupancy rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
OUTPUT_PATH = "parking_prediction.csv"
prediction_df.to_csv(OUTPUT_PATH, index=False)
print(f"Forecast results saved to: {OUTPUT_PATH}")